# Performance Analysis (perf_001)

Wall-clock runtimes and scaling analysis for the 8 representative cases
across worker counts W = 1, 2, 4, 8, 16.

In [ ]:
import sys
from pathlib import Path

import msgpack
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

sys.path.insert(0, str(Path("..").resolve()))
sys.path.insert(0, str(Path("../../cost_001").resolve()))

from load_results import load_results_raw
from ship_routing.app import RoutingResult
from experiment_params import REPRESENTATIVE_JOURNEYS, REPRESENTATIVE_CASES

## 1. Load results

In [ ]:
result_files = sorted(Path("../results").glob("perf_*.msgpack"))
print(f"Found {len(result_files)} result files")

# Load all into a single dict, dedup by key
raw = {}
for f in result_files:
    with open(f, "rb") as fh:
        raw.update(msgpack.unpack(fh, raw=False))
print(f"Unique results: {len(raw)}")

In [ ]:
# Parse keys into structured records
# Key format: perf:c{case}:r{replica}:w{workers}:seed{seed}
records = []
for key, value in tqdm(raw.items(), desc="parsing"):
    parts = key.split(":")
    case_idx = int(parts[1][1:])
    replica_idx = int(parts[2][1:])
    workers = int(parts[3][1:])
    seed = int(parts[4][4:])

    rr = RoutingResult.from_msgpack(value)
    df = rr.logs.to_dataframe()
    t0 = df.timestamp.min()
    t1 = df.timestamp.max()

    # Stage boundaries from log timestamps
    stages = df.groupby("stage").timestamp.agg(["min", "max"])

    # Total runtime
    total_s = (t1 - t0).total_seconds()

    # Stage durations (using stage transition boundaries)
    t_load = (stages.loc["initialization", "min"] - stages.loc["load_forcing", "min"]).total_seconds()
    t_init = (stages.loc["warmup", "min"] - stages.loc["initialization", "min"]).total_seconds()
    t_warmup = (stages.loc["ga_mutation", "min"] - stages.loc["warmup", "min"]).total_seconds()
    t_genetic = (stages.loc["gradient_polishing", "min"] - stages.loc["ga_mutation", "min"]).total_seconds()
    t_gd = (stages.loc["gd_iteration", "max"] - stages.loc["gradient_polishing", "min"]).total_seconds()

    # Journey label
    rc = REPRESENTATIVE_CASES[case_idx]
    label = f"{rc['route'].split('_')[1][:3].title()} {rc['month']:02d} {rc['speed']:.0f}kn"

    records.append({
        "key": key,
        "case_idx": case_idx,
        "replica_idx": replica_idx,
        "workers": workers,
        "seed": seed,
        "label": label,
        "total_s": total_s,
        "t_load": t_load,
        "t_init": t_init,
        "t_warmup": t_warmup,
        "t_genetic": t_genetic,
        "t_gd": t_gd,
    })

perf = pd.DataFrame.from_records(records)
print(f"{len(perf)} records loaded")
perf.head()

In [ ]:
# Coverage check
coverage = perf.groupby(["case_idx", "workers"]).replica_idx.count().unstack(fill_value=0)
coverage.columns = [f"w{w:02d}" for w in coverage.columns]
coverage.index = [REPRESENTATIVE_CASES[i] for i in coverage.index]
print("Replicas per (case, workers):")
coverage

## 2. Wall-clock time vs workers

In [ ]:
# Median runtime (min) pivoted: cases as columns, workers as index
runtime_pivot = (
    perf.groupby(["label", "workers"]).total_s.median().unstack("label") / 60
)
runtime_pivot.plot(style="o-", ylabel="Runtime (min)", xlabel="Workers")
plt.show()

## 3. Stage breakdown

In [ ]:
stage_cols = ["t_load", "t_init", "t_warmup", "t_genetic", "t_gd"]
stage_labels = ["Data loading", "Initialization", "Warmup", "Genetic", "Gradient descent"]

stage_median = perf.groupby(["label", "workers"])[stage_cols].median()

for w in [1, 8]:
    if w not in perf.workers.values:
        continue
    sub = stage_median.xs(w, level="workers") / 60
    sub.columns = stage_labels
    sub.plot.barh(stacked=True, title=f"Stage breakdown (W={w})", xlabel="Time (min)")
    plt.show()

## 4. Speedup and parallel efficiency

In [ ]:
pivot = perf.groupby(["case_idx", "label", "workers"]).total_s.median().reset_index()
pivot_wide = pivot.pivot(index=["case_idx", "label"], columns="workers", values="total_s")

t1 = pivot_wide[1]

# Speedup = T1 / TW
speedup = pivot_wide.div(t1, axis=0).rdiv(1)
speedup.columns = [f"S(W={w})" for w in speedup.columns]

# Efficiency = speedup / W
efficiency = pivot_wide.copy()
for w in efficiency.columns:
    efficiency[w] = t1 / (w * efficiency[w])
efficiency.columns = [f"E(W={w})" for w in efficiency.columns]

# Runtime in minutes
runtime_min = (pivot_wide / 60).round(1)
runtime_min.columns = [f"T(W={w}) min" for w in runtime_min.columns]

display(runtime_min)
print()
display(speedup.round(2))
print()
display(efficiency.round(2))

In [ ]:
# Speedup as DataFrame, workers as index, cases as columns
speedup_df = speedup.T
speedup_df.index = [int(c.split("=")[1].rstrip(")")) for c in speedup_df.index]
speedup_df.columns = [f"C{ci}: {lab}" for ci, lab in speedup_df.columns]
speedup_df.plot(style="o-", ylabel="Speedup", xlabel="Workers", title="Parallel speedup")
plt.plot([1, 16], [1, 16], "k--", label="ideal")
plt.legend()
plt.show()